<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [2]:
%pip install -U -q keras-hub keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.27.1 which is incompatible.


In [3]:
import keras
import keras_hub
import numpy as np

In [4]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_270m")

100%|██████████| 965/965 [00:00<00:00, 1.90MB/s]


100%|██████████| 3.23k/3.23k [00:00<00:00, 2.65MB/s]


100%|██████████| 4.47M/4.47M [00:02<00:00, 2.25MB/s]


100%|██████████| 512M/512M [00:34<00:00, 15.6MB/s]


In [5]:
gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

'I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swoll

In [10]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to pandas and sample
df = ds.to_pandas().sample(200, random_state=42)

features = {
    "prompts": df["input"].tolist(),
    "responses": df["output"].tolist()
}

In [11]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

200
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.
Dear patient Here are the possibilities of what you might have.1)PhlebitisPhlebitis means inflammation of the veins, and can cause redness, itching, irritation, pain, and swelling. A simple Doppler can rule this out.2Blood clot in the lifeblood clots in the leg can become very dangerous, symptoms include swelling, redness, tenderness in the leg. Coagulation profile with an angiography may be required3)Cellulitis


In [16]:
gemma_lm.backbone.enable_lora(rank=4)

gemma_lm.preprocessor.sequence_length = 128

In [17]:
gemma_lm.fit(features, epochs=1, batch_size=8)

ValueError: Unknown variable: <Variable path=decoder_block_0/attention/value/kernel, shape=(1, 640, 256), dtype=float32>. This optimizer can only be called for the variables it was originally built with. When working with a new set of variables, you should recreate a new optimizer instance.

In [ ]:
gemma_lm.generate("I have been having headaches every day for the past week. What could be causing this and what should I do?", max_length=500)

In [ ]:
gemma_lm.load_lora_weights("lora_weights.h5")